In [0]:
from pyspark.sql.functions import col

df_oc = spark.table(
    "chilecompra.silver.ordenes_compra"
)

In [0]:
df_compras_validas = (
    df_oc
    .filter(
        col("fecha_envio").isNotNull()
        & col("monto_total_clp").isNotNull()
        & (col("codigo_estado") != 9)
    )
)

In [0]:
from pyspark.sql.functions import (
    year,
    month,
    countDistinct,
    sum,
    avg
)

df_compras_mensuales = (
    df_compras_validas
    .groupBy(
        year(col("fecha_envio")).alias("anio"),
        month(col("fecha_envio")).alias("mes")
    )
    .agg(
        countDistinct("codigo").alias("cantidad_ordenes"),
        sum("monto_total_clp").alias("monto_total_ordenes_clp"),
        avg("monto_total_clp").alias("monto_promedio_orden_clp"),
        countDistinct("codigo_proveedor").alias("proveedores_distintos"),
        countDistinct("codigo_organismo_publico").alias("organismos_distintos")
    )
    .orderBy("anio", "mes")
)


In [0]:
gold_rows = df_compras_mensuales.count()

null_keys = (
    df_compras_mensuales
    .filter(
        col("anio").isNull() |
        col("mes").isNull()
    )
    .count()
)

negative_amounts = (
    df_compras_mensuales
    .filter(col("monto_total_ordenes_clp") < 0)
    .count()
)

if gold_rows == 0:
    raise ValueError("DQ FAILED: Gold aggregation produced 0 rows")

if null_keys > 0:
    raise ValueError("DQ FAILED: Gold has NULL year/month")

if negative_amounts > 0:
    raise ValueError("DQ FAILED: Gold has negative monthly amounts")

print(f"Pre-write Gold DQ passed: {gold_rows} months")

In [0]:
target_table = "chilecompra.gold.compras_mensuales"

(
    df_compras_mensuales
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

In [0]:
df_gold = spark.table(target_table)

gold_rows = df_gold.count()

gold_distinct_months = (
    df_gold
    .select("anio", "mes")
    .distinct()
    .count()
)

if gold_rows != gold_distinct_months:
    raise ValueError(
        "DQ FAILED: duplicated year/month in gold.compras_mensuales"
    )

if gold_rows != df_compras_mensuales.count():
    raise ValueError(
        "DQ FAILED: Gold row count does not match source aggregation"
    )

print(
    f"Gold compras_mensuales DQ passed: {gold_rows} months"
)